# **BizFlow360 Dataset Unification**

This notebook aims to unite two real-world datasets into one, perform EDA and provide a clean structured dataset\
we can use for our model training

# 1. Setup and Configuration
**What this cell does:** Imports the libraries and sets the correct file paths (going up TWO levels to reach the project root) to load Denis's three cleaned datasets.

In [24]:
import pandas as pd
import numpy as np
import os

# Go up TWO levels to reach the root BizFlow360 folder
base_dir = os.path.abspath('../..')
data_dir = os.path.join(base_dir, 'data_eda', 'processed')

print(f"Loading data from: {data_dir}\n")

df_modeling = pd.read_csv(os.path.join(data_dir, 'msme_financial_distress_modeling_dataset.csv'))
df_analysis = pd.read_csv(os.path.join(data_dir, 'msme_analysis_clean.csv'))
df_world_bank = pd.read_csv(os.path.join(data_dir, 'world_bank', 'world_bank_modeling_ready.csv'))

print(f"✅ Modeling Dataset (Target & Ratios): {df_modeling.shape}")
print(f"✅ Analysis Dataset (Features & Demographics): {df_analysis.shape}")
print(f"✅ World Bank Dataset (Macro Context - kept for EDA only): {df_world_bank.shape}")

Loading data from: /home/mikel/BizFlow360/data_eda/processed

✅ Modeling Dataset (Target & Ratios): (15922, 18)
✅ Analysis Dataset (Features & Demographics): (20596, 26)
✅ World Bank Dataset (Macro Context - kept for EDA only): (159, 55)


# 2. Initial Inspection
**What this cell does:** Confirms that the modeling dataset has NO ID column, while the analysis dataset uses 'key2' (the UUID). This proves we cannot merge on an ID and must use a "financial fingerprint" instead.

In [25]:
print("--- Modeling Dataset Columns ---")
print(df_modeling.columns.tolist())
print("\n--- Analysis Dataset Columns ---")
print(df_analysis.columns.tolist())

print("\n⚠️ Note: The modeling dataset has no UUID. We will align rows using the")
print("shared financial values (eh01_1, eh15_1, eh04_1, eh09) as a fingerprint.")

--- Modeling Dataset Columns ---
['eh01_1', 'eh15_1', 'eh04_1', 'eh09', 'en01', 'en01b', 'eh01_1_clean', 'eh15_1_clean', 'eh04_1_clean', 'net_income_margin', 'revenue_change_ratio', 'business_closed', 'number_closed_establishments', 'revenue_decline', 'zero_or_missing_net_income', 'low_revenue', 'financial_risk_score', 'business_performance_score']

--- Analysis Dataset Columns ---
['key2', 'county11', 'eb01_2', 'eb03', 'eb04', 'eb05', 'eb16_1', 'ec01', 'ec02', 'eg03_1', 'eg04_1', 'eg05_1', 'eg06_1', 'eg07_1', 'eg08_1', 'eg09_1', 'eg10_1', 'eg22_1', 'eh01_1', 'eh02_1', 'eh03_1', 'eh04_1', 'eh09', 'eh15_1', 'eh22_1', 'total_selected_expenses']

⚠️ Note: The modeling dataset has no UUID. We will align rows using the
shared financial values (eh01_1, eh15_1, eh04_1, eh09) as a fingerprint.


# 3. Merging via Financial Fingerprint
**What this cell does:** Aligns the two KNBS datasets using the raw financial values that exist in BOTH files. A duplicate counter ('_dup_id') ensures that repeated identical fingerprints match one-to-one instead of exploding into duplicate rows. Rows with no performance ranking ('N/A') are removed first.

In [26]:
# 1. Remove rows with no valid performance ranking (they have no usable target)
df_modeling = df_modeling[~df_modeling['eh09'].isin(['N/A (business younger than 1 month)'])].copy()
df_modeling = df_modeling[df_modeling['eh09'].notna()].copy()

# 2. Force numeric keys and replace NaNs with sentinels so missing values can match
shared_keys = ['eh01_1', 'eh15_1', 'eh04_1', 'eh09']

for col in ['eh01_1', 'eh15_1', 'eh04_1']:
    df_modeling[col] = pd.to_numeric(df_modeling[col], errors='coerce').fillna(-9999)
    df_analysis[col] = pd.to_numeric(df_analysis[col], errors='coerce').fillna(-9999)

df_modeling['eh09'] = df_modeling['eh09'].fillna('MISSING')
df_analysis['eh09'] = df_analysis['eh09'].fillna('MISSING')

# 3. Add a duplicate counter so identical fingerprints match 1-to-1 (no row explosion)
df_modeling['_dup_id'] = df_modeling.groupby(shared_keys).cumcount()
df_analysis['_dup_id'] = df_analysis.groupby(shared_keys).cumcount()

# 4. Perform the merge
df_unified = pd.merge(df_modeling, df_analysis, on=shared_keys + ['_dup_id'], how='inner')

# 5. Clean up sentinels and helper column
for col in ['eh01_1', 'eh15_1', 'eh04_1']:
    df_unified[col] = df_unified[col].replace(-9999, np.nan)
df_unified['eh09'] = df_unified['eh09'].replace('MISSING', np.nan)
df_unified.drop(columns=['_dup_id'], inplace=True)

print(f"✅ Merge successful! Unified Dataset Shape: {df_unified.shape}")

# 6. Define the binary target: 'Bad' performance = financially distressed
df_unified['distress_label'] = (df_unified['eh09'] == 'Bad').astype(int)
print("\nTarget distribution (0 = Stable, 1 = Distressed):")
print(df_unified['distress_label'].value_counts())

✅ Merge successful! Unified Dataset Shape: (15814, 40)

Target distribution (0 = Stable, 1 = Distressed):
distress_label
0    10039
1     5775
Name: count, dtype: int64


In [27]:
df_unified

,eh01_1,eh15_1,eh04_1,eh09,en01,en01b,eh01_1_clean,eh15_1_clean,eh04_1_clean,net_income_margin,...,eg07_1,eg08_1,eg09_1,eg10_1,eg22_1,eh02_1,eh03_1,eh22_1,total_selected_expenses,distress_label
0,0.0,7500.0,10000.0,Good,No,NaN,0.0,7500.0,10000.0,1.333333,...,30.00,0.0,0.0,0.0,0.0,0.0,0.0,NaN,570.00,0
1,39000.0,9750.0,50000.0,Bad,No,NaN,39000.0,9750.0,50000.0,5.128205,...,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,2242.50,1
2,6825.0,6825.0,NaN,Normal,No,NaN,6825.0,6825.0,NaN,NaN,...,97.50,0.0,0.0,0.0,0.0,NaN,NaN,NaN,2254.20,0
3,2925.0,2925.0,5000.0,Normal,No,NaN,2925.0,2925.0,5000.0,1.709402,...,0.00,0.0,0.0,0.0,0.0,30000.0,50000.0,NaN,5.85,0
4,133380.0,126750.0,120000.0,Good,No,NaN,133380.0,126750.0,120000.0,0.946746,...,2925.00,0.0,0.0,0.0,0.0,488000.0,500000.0,NaN,10130.25,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15809,14040.0,12675.0,7200.0,Normal,No,NaN,14040.0,12675.0,7200.0,0.568047,...,0.00,0.0,0.0,0.0,0.0,90000.0,100000.0,NaN,236.73,0
15810,6240.0,6240.0,4000.0,Normal,No,NaN,6240.0,6240.0,4000.0,0.641026,...,97.50,0.0,0.0,0.0,0.0,28000.0,20000.0,NaN,624.00,0
15811,1170.0,1170.0,2000.0,Normal,Yes,1.0,1170.0,1170.0,2000.0,1.709402,...,39.00,0.0,0.0,0.0,0.0,10000.0,4000.0,NaN,370.50,0
15812,22500.0,4500.0,15000.0,Normal,No,NaN,22500.0,4500.0,15000.0,3.333333,...,75.00,0.0,0.0,0.0,0.0,0.0,0.0,NaN,315.00,0


# 4. Renaming Columns to Plain English
**What this cell does:** Translates the cryptic KNBS survey codes into readable English names using Denis's metadata file, and drops ID columns the model doesn't need.

In [29]:
column_mapping = {
    # Demographics
    'county11': 'county',
    'eb01_2': 'sector',
    'ec01': 'male_working_owners',
    'ec02': 'female_working_owners',
    # Expenditure breakdown
    'eg03_1': 'monthly_rent_expense',
    'eg05_1': 'monthly_electricity_expense',
    'eg10_1': 'monthly_credit_expense',
    'eg22_1': 'monthly_social_responsibility_expense',
    'total_selected_expenses': 'total_monthly_expenses',
    # Financials
    'eh01_1': 'revenue_last_month',
    'eh02_1': 'stock_value_beginning',
    'eh03_1': 'stock_value_end',
    'eh04_1': 'net_income_last_month',
    'eh09': 'business_performance',
    'eh15_1': 'normal_monthly_revenue',
    'eh22_1': 'total_turnover_2015',
    'eh01_1_clean': 'revenue_last_month_clean',
    'eh15_1_clean': 'normal_monthly_revenue_clean',
    'eh04_1_clean': 'net_income_last_month_clean',
    # Closure history
    'en01': 'closed_establishment_past_5_years',
    'en01b': 'number_closed_establishments',
}

df_unified.rename(columns=column_mapping, inplace=True)

# Drop ID and redundant text columns (binary 'business_closed' already exists)
cols_to_drop = ['key2', 'closed_establishment_past_5_years',
                'business_performance_score', 'financial_risk_score']
df_unified.drop(columns=[c for c in cols_to_drop if c in df_unified.columns], inplace=True)

print("✅ Columns renamed to English and redundant IDs dropped!")

✅ Columns renamed to English and redundant IDs dropped!


In [30]:
df_unified

,revenue_last_month,normal_monthly_revenue,net_income_last_month,business_performance,number_closed_establishments,revenue_last_month_clean,normal_monthly_revenue_clean,net_income_last_month_clean,net_income_margin,revenue_change_ratio,...,eg07_1,eg08_1,eg09_1,monthly_credit_expense,monthly_social_responsibility_expense,stock_value_beginning,stock_value_end,total_turnover_2015,total_monthly_expenses,distress_label
0,0.0,7500.0,10000.0,Good,NaN,0.0,7500.0,10000.0,1.333333,0.000000,...,30.00,0.0,0.0,0.0,0.0,0.0,0.0,NaN,570.00,0
1,39000.0,9750.0,50000.0,Bad,NaN,39000.0,9750.0,50000.0,5.128205,4.000000,...,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,2242.50,1
2,6825.0,6825.0,NaN,Normal,NaN,6825.0,6825.0,NaN,NaN,1.000000,...,97.50,0.0,0.0,0.0,0.0,NaN,NaN,NaN,2254.20,0
3,2925.0,2925.0,5000.0,Normal,NaN,2925.0,2925.0,5000.0,1.709402,1.000000,...,0.00,0.0,0.0,0.0,0.0,30000.0,50000.0,NaN,5.85,0
4,133380.0,126750.0,120000.0,Good,NaN,133380.0,126750.0,120000.0,0.946746,1.052308,...,2925.00,0.0,0.0,0.0,0.0,488000.0,500000.0,NaN,10130.25,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15809,14040.0,12675.0,7200.0,Normal,NaN,14040.0,12675.0,7200.0,0.568047,1.107692,...,0.00,0.0,0.0,0.0,0.0,90000.0,100000.0,NaN,236.73,0
15810,6240.0,6240.0,4000.0,Normal,NaN,6240.0,6240.0,4000.0,0.641026,1.000000,...,97.50,0.0,0.0,0.0,0.0,28000.0,20000.0,NaN,624.00,0
15811,1170.0,1170.0,2000.0,Normal,1.0,1170.0,1170.0,2000.0,1.709402,1.000000,...,39.00,0.0,0.0,0.0,0.0,10000.0,4000.0,NaN,370.50,0
15812,22500.0,4500.0,15000.0,Normal,NaN,22500.0,4500.0,15000.0,3.333333,5.000000,...,75.00,0.0,0.0,0.0,0.0,0.0,0.0,NaN,315.00,0


# 5. Cleaning KNBS Placeholders and Text Errors
**What this cell does:** KNBS uses negative codes (like -19.11) to mean "Missing". This cell converts them to real NaNs and forces every financial column to be a proper number (turning text like "Good"/"Bad" in numeric columns into NaN).

In [32]:
# ==========================================
# CLEAN KNBS PLACEHOLDERS & FORCE NUMERIC
# ==========================================

# 1. Drop any duplicate columns that might have slipped through the merge
# This keeps only the first occurrence of any duplicated column name
df_unified = df_unified.loc[:, ~df_unified.columns.duplicated()]

knbs_placeholders = [-19.11, -19.305, -2.4696, -2.4948, -20.58, -73.5, -74.25, -3.528, -8.82]

# List of columns that SHOULD be numbers (updated to match the English names from Cell 8)
numeric_cols = [
    'male_working_owners', 'female_working_owners', 'total_monthly_expenses',
    'monthly_rent_expense', 'monthly_electricity_expense', 'monthly_credit_expense',
    'monthly_social_responsibility_expense', 'revenue_last_month', 'normal_monthly_revenue',
    'net_income_last_month', 'stock_value_beginning', 'stock_value_end',
    'total_turnover_2015', 'net_income_margin', 'revenue_change_ratio',
    'number_closed_establishments'
]

for col in numeric_cols:
    if col in df_unified.columns:
        # 1. Replace KNBS placeholders with actual NaN
        df_unified[col] = df_unified[col].replace(knbs_placeholders, np.nan)
        
        # 2. Force to numeric, turning "Bad", "Good", "N/A..." into NaN
        df_unified[col] = pd.to_numeric(df_unified[col], errors='coerce')

print("✅ Duplicate columns removed, KNBS placeholders cleared, and columns forced to numeric types.")

✅ Duplicate columns removed, KNBS placeholders cleared, and columns forced to numeric types.


In [33]:
df_unified

,revenue_last_month,normal_monthly_revenue,net_income_last_month,business_performance,number_closed_establishments,revenue_last_month_clean,normal_monthly_revenue_clean,net_income_last_month_clean,net_income_margin,revenue_change_ratio,...,eg07_1,eg08_1,eg09_1,monthly_credit_expense,monthly_social_responsibility_expense,stock_value_beginning,stock_value_end,total_turnover_2015,total_monthly_expenses,distress_label
0,0.0,7500.0,10000.0,Good,NaN,0.0,7500.0,10000.0,1.333333,0.000000,...,30.00,0.0,0.0,0.0,0.0,0.0,0.0,NaN,570.00,0
1,39000.0,9750.0,50000.0,Bad,NaN,39000.0,9750.0,50000.0,5.128205,4.000000,...,0.00,0.0,0.0,0.0,0.0,NaN,NaN,NaN,2242.50,1
2,6825.0,6825.0,NaN,Normal,NaN,6825.0,6825.0,NaN,NaN,1.000000,...,97.50,0.0,0.0,0.0,0.0,NaN,NaN,NaN,2254.20,0
3,2925.0,2925.0,5000.0,Normal,NaN,2925.0,2925.0,5000.0,1.709402,1.000000,...,0.00,0.0,0.0,0.0,0.0,30000.0,50000.0,NaN,5.85,0
4,133380.0,126750.0,120000.0,Good,NaN,133380.0,126750.0,120000.0,0.946746,1.052308,...,2925.00,0.0,0.0,0.0,0.0,488000.0,500000.0,NaN,10130.25,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15809,14040.0,12675.0,7200.0,Normal,NaN,14040.0,12675.0,7200.0,0.568047,1.107692,...,0.00,0.0,0.0,0.0,0.0,90000.0,100000.0,NaN,236.73,0
15810,6240.0,6240.0,4000.0,Normal,NaN,6240.0,6240.0,4000.0,0.641026,1.000000,...,97.50,0.0,0.0,0.0,0.0,28000.0,20000.0,NaN,624.00,0
15811,1170.0,1170.0,2000.0,Normal,1.0,1170.0,1170.0,2000.0,1.709402,1.000000,...,39.00,0.0,0.0,0.0,0.0,10000.0,4000.0,NaN,370.50,0
15812,22500.0,4500.0,15000.0,Normal,NaN,22500.0,4500.0,15000.0,3.333333,5.000000,...,75.00,0.0,0.0,0.0,0.0,0.0,0.0,NaN,315.00,0


# 6. Imputing Missing Values (NaNs)
**What this cell does:** Machine learning models cannot handle NaNs. This cell fills missing numbers with the median and missing categories with the most frequent value (mode).

In [34]:
for col in numeric_cols:
    if col in df_unified.columns:
        median_val = df_unified[col].median()
        df_unified[col] = df_unified[col].fillna(median_val if not pd.isna(median_val) else 0)

categorical_cols = ['county', 'sector']
for col in categorical_cols:
    if col in df_unified.columns:
        df_unified[col] = df_unified[col].fillna(df_unified[col].mode()[0])

remaining_nans = df_unified.isnull().sum().sum()
print(f"✅ Imputation complete! Remaining NaNs in dataset: {remaining_nans}")

✅ Imputation complete! Remaining NaNs in dataset: 11083


In [35]:
df_unified

,revenue_last_month,normal_monthly_revenue,net_income_last_month,business_performance,number_closed_establishments,revenue_last_month_clean,normal_monthly_revenue_clean,net_income_last_month_clean,net_income_margin,revenue_change_ratio,...,eg07_1,eg08_1,eg09_1,monthly_credit_expense,monthly_social_responsibility_expense,stock_value_beginning,stock_value_end,total_turnover_2015,total_monthly_expenses,distress_label
0,0.0,7500.0,10000.0,Good,1.0,0.0,7500.0,10000.0,1.333333,0.000000,...,30.00,0.0,0.0,0.0,0.0,0.0,0.0,288000.0,570.00,0
1,39000.0,9750.0,50000.0,Bad,1.0,39000.0,9750.0,50000.0,5.128205,4.000000,...,0.00,0.0,0.0,0.0,0.0,20000.0,10000.0,288000.0,2242.50,1
2,6825.0,6825.0,20000.0,Normal,1.0,6825.0,6825.0,NaN,2.051282,1.000000,...,97.50,0.0,0.0,0.0,0.0,20000.0,10000.0,288000.0,2254.20,0
3,2925.0,2925.0,5000.0,Normal,1.0,2925.0,2925.0,5000.0,1.709402,1.000000,...,0.00,0.0,0.0,0.0,0.0,30000.0,50000.0,288000.0,5.85,0
4,133380.0,126750.0,120000.0,Good,1.0,133380.0,126750.0,120000.0,0.946746,1.052308,...,2925.00,0.0,0.0,0.0,0.0,488000.0,500000.0,288000.0,10130.25,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15809,14040.0,12675.0,7200.0,Normal,1.0,14040.0,12675.0,7200.0,0.568047,1.107692,...,0.00,0.0,0.0,0.0,0.0,90000.0,100000.0,288000.0,236.73,0
15810,6240.0,6240.0,4000.0,Normal,1.0,6240.0,6240.0,4000.0,0.641026,1.000000,...,97.50,0.0,0.0,0.0,0.0,28000.0,20000.0,288000.0,624.00,0
15811,1170.0,1170.0,2000.0,Normal,1.0,1170.0,1170.0,2000.0,1.709402,1.000000,...,39.00,0.0,0.0,0.0,0.0,10000.0,4000.0,288000.0,370.50,0
15812,22500.0,4500.0,15000.0,Normal,1.0,22500.0,4500.0,15000.0,3.333333,5.000000,...,75.00,0.0,0.0,0.0,0.0,0.0,0.0,288000.0,315.00,0


# 7. Final Feature Selection and Export
**What this cell does:** Selects only the relevant, non-leaky features for the models, attaches the target variable, and saves the clean unified dataset to ml_models/data/ so notebooks 10-15 can train on REAL data.

In [36]:
final_features = [
    'county', 'sector', 'male_working_owners', 'female_working_owners',
    'total_monthly_expenses', 'monthly_rent_expense', 'monthly_electricity_expense',
    'monthly_credit_expense', 'monthly_social_responsibility_expense',
    'revenue_last_month', 'normal_monthly_revenue', 'net_income_last_month',
    'stock_value_beginning', 'stock_value_end', 'total_turnover_2015',
    'net_income_margin', 'revenue_change_ratio', 'business_closed',
    'number_closed_establishments', 'revenue_decline',
    'zero_or_missing_net_income', 'low_revenue'
]

available_features = [col for col in final_features if col in df_unified.columns]

df_final = df_unified[available_features + ['distress_label']].copy()

# Save to ml_models/data/
output_path = os.path.join(base_dir, 'ml_models', 'data', 'unified_msme_modeling_data.csv')
df_final.to_csv(output_path, index=False)

print(f"🏆 SUCCESS! Unified dataset saved to: {output_path}")
print(f"Final shape: {df_final.shape} | Features: {len(available_features)}")
print("\nReady to train models in notebooks 10-15!")

🏆 SUCCESS! Unified dataset saved to: /home/mikel/BizFlow360/ml_models/data/unified_msme_modeling_data.csv
Final shape: (15814, 23) | Features: 22

Ready to train models in notebooks 10-15!
